<a href="https://colab.research.google.com/github/byter66/Aegis/blob/main/landmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://s3.amazonaws.com/google-landmark/metadata/train.csv
!wget https://s3.amazonaws.com/google-landmark/metadata/train_label_to_category.csv


--2026-10-09 06:04:52--  https://s3.amazonaws.com/google-landmark/metadata/train.csv
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.82.38, 52.217.232.184, 52.217.121.120, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.82.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 525832518 (501M) [text/csv]
Saving to: ‘train.csv’

train.csv           100%[===================>] 501.47M  37.0MB/s    in 16s     

2026-10-09 06:05:09 (30.5 MB/s) - ‘train.csv’ saved [525832518/525832518]

--2026-10-09 06:05:09--  https://s3.amazonaws.com/google-landmark/metadata/train_label_to_category.csv
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.212.166, 16.15.237.165, 16.15.254.138, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.212.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15153105 (14M) [text/csv]
Saving to: ‘train_label_to_category.csv’

train_label_to_cate 100%[===================>]  14.45M  18.0MB/s

In [ ]:
import pandas as pd

cats = pd.read_csv('train_label_to_category.csv')
known = cats[cats['category'].str.contains('Eiffel|Big_Ben|Golden_Gate', case=False, na=False)]

train = pd.read_csv('train.csv')
subset = train[train['landmark_id'].isin(known['landmark_id'])].sample(10)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
path = "/content/drive/MyDrive/ColabNotebooks/landmark_images/83a1b2cbbf22a3c3.jpg"
print(os.path.getsize(path), "bytes")

with open(path, 'rb') as f:
    print(f.read(200))

126 bytes
b'Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.\n'


In [ ]:
!wget https://s3.amazonaws.com/google-landmark/train/images_000.tar
!mkdir -p /content/drive/MyDrive/Colab\ Notebooks/landmark_images
!tar -xf images_000.tar -C /content/drive/MyDrive/Colab\ Notebooks/landmark_images

--2026-10-06 17:30:52--  https://s3.amazonaws.com/google-landmark/train/images_000.tar
Resolving s3.amazonaws.com (s3.amazonaws.com)... 54.231.134.192, 16.15.236.129, 16.15.245.230, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|54.231.134.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1067018752 (1018M) [application/x-tar]
Saving to: ‘images_000.tar’

images_000.tar      100%[===================>]   1018M  19.9MB/s    in 54s     

2026-10-06 17:31:47 (18.8 MB/s) - ‘images_000.tar’ saved [1067018752/1067018752]



In [ ]:
import os
os.makedirs('/content/drive/MyDrive/ColabNotebooks/landmark_images', exist_ok=True)

for i, row in subset.iterrows():
    try:
        r = requests.get(row['url'], timeout=10)
        if r.status_code == 200 and r.headers.get('Content-Type', '').startswith('image'):
            path = f"/content/drive/MyDrive/ColabNotebooks/landmark_images/{row['id']}.jpg"
            with open(path, 'wb') as f:
                f.write(r.content)
        else:
            print(f"Skipped {row['id']}: status {r.status_code}, type {r.headers.get('Content-Type')}")
    except Exception as e:
        print(f"Failed on {row['id']}: {e}")

Skipped 981cb5d8c18c34a2: status 403, type text/plain
Skipped 48b0a687c4fcc059: status 403, type text/plain
Skipped a1d5e890f4d0fc7a: status 403, type text/plain
Skipped 8ad3000266d0d8f2: status 403, type text/plain
Skipped 01e450be8d438bf5: status 403, type text/plain
Skipped b4ff8d9c6e6c45dd: status 403, type text/plain
Skipped 6cac34590c47c28c: status 403, type text/plain
Skipped 843767c368ca82cc: status 403, type text/plain
Skipped bd21d5e3a595d479: status 403, type text/plain
Skipped a281716596d88b63: status 403, type text/plain


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np

In [ ]:
model = ResNet50(weights='imagenet')

NameError: name 'ResNet50' is not defined

In [ ]:
img_path = "/content/drive/MyDrive/Colab Notebooks/landmark_images/0/0/3/003af62c46245944.jpg"  # use a real path from your downloaded images

img = image.load_img(img_path, target_size=(224, 224))  # ResNet50 expects 224x224
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

predictions = model.predict(img_array)
print(decode_predictions(predictions, top=3)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
[('n09193705', 'alp', np.float32(0.87645227)), ('n09468604', 'valley', np.float32(0.031179318)), ('n03792972', 'mountain_tent', np.float32(0.027680436))]


In [ ]:
def predict_image(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = np.expand_dims(image.img_to_array(img), axis=0)
    img_array = preprocess_input(img_array)
    preds = model.predict(img_array)
    return decode_predictions(preds, top=3)[0]

In [ ]:
import pandas as pd

cats = pd.read_csv('train_label_to_category.csv')

known = cats[cats['category'].str.contains('labcoat|megalith|church', case=False, na=False)]
print(known)

        landmark_id                                           category
18               18  http://commons.wikimedia.org/wiki/Category:Ort...
37               37  http://commons.wikimedia.org/wiki/Category:Chu...
58               58  http://commons.wikimedia.org/wiki/Category:Sai...
60               60  http://commons.wikimedia.org/wiki/Category:Ill...
62               62  http://commons.wikimedia.org/wiki/Category:Chu...
...             ...                                                ...
203048       203048  http://commons.wikimedia.org/wiki/Category:Eli...
203052       203052  http://commons.wikimedia.org/wiki/Category:Mox...
203078       203078  http://commons.wikimedia.org/wiki/Category:Aga...
203083       203083  http://commons.wikimedia.org/wiki/Category:St....
203085       203085  http://commons.wikimedia.org/wiki/Category:Chu...

[13097 rows x 2 columns]


In [ ]:
import os

base = "/content/drive/MyDrive/Colab Notebooks/landmark_images"
downloaded_ids = {}
for root, dirs, files in os.walk(base):
    for f in files:
        if f.endswith('.jpg'):
            img_id = f.replace('.jpg', '')
            downloaded_ids[img_id] = os.path.join(root, f)

print(f"Found {len(downloaded_ids)} downloaded images")

Found 8267 downloaded images


In [ ]:
train = pd.read_csv('train.csv')

matches = train[train['id'].isin(downloaded_ids.keys()) & train['landmark_id'].isin(known['landmark_id'])]
print(f"Found {len(matches)} matching images across your chosen landmarks")
print(matches['landmark_id'].value_counts())

Found 425 matching images across your chosen landmarks
landmark_id
103827    3
107164    3
69835     2
21843     2
181508    2
         ..
85018     1
202433    1
173890    1
171358    1
68121     1
Name: count, Length: 400, dtype: int64


In [ ]:
import shutil

sorted_base = "/content/drive/MyDrive/Colab Notebooks/landmark_images_sorted"

id_to_name = dict(zip(known['landmark_id'], known['category'].str.extract(r'/([^/]+)$')[0]))

for _, row in matches.iterrows():
    landmark_id = row['landmark_id']
    img_id = row['id']
    class_name = id_to_name.get(landmark_id, str(landmark_id))

    dest_dir = os.path.join(sorted_base, class_name)
    os.makedirs(dest_dir, exist_ok=True)

    src_path = downloaded_ids[img_id]
    dest_path = os.path.join(dest_dir, f"{img_id}.jpg")
    shutil.copy(src_path, dest_path)

print("Done sorting.")

Done sorting.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

datagen = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)

train_data = datagen.flow_from_directory(
    "/content/drive/MyDrive/Colab Notebooks/landmark_images",
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    "/content/drive/MyDrive/Colab Notebooks/landmark_images",
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='validation'
)

Found 6613 images belonging to 1 classes.
Found 1653 images belonging to 1 classes.


In [ ]:

from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_data, validation_data=val_data, epochs=10)

Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


414/414 ━━━━━━━━━━━━━━━━━━━━ 2541s 6s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/10
414/414 ━━━━━━━━━━━━━━━━━━━━ 71s 172ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/10
414/414 ━━━━━━━━━━━━━━━━━━━━ 75s 181ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/10
414/414 ━━━━━━━━━━━━━━━━━━━━ 75s 181ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/10
414/414 ━━━━━━━━━━━━━━━━━━━━ 80s 193ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/10
414/414 ━━━━━━━━━━━━━━━━━━━━ 74s 177ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/10
414/414 ━━━━━━━━━━━━━━━━━━━━ 79s 191ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/10
414/414 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
print(history.history['val_accuracy'])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
